In [1]:
import matplotlib.pyplot as plt
import sys
import os
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import tqdm
import libsbml
import pickle

In [2]:
fname = "/home/eka/thesis/ParkRam2/ParkRam_constant_feed.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

In [3]:
def convert_symbol_to_label(symbol):
    return f"{symbol}_noise"

def convert_symbol_to_sd(symbol):
    return f"sd_{symbol}"

In [4]:
maxes = {}
mins = {}
maxes_to_sbml = {}
mins_to_sbml = {}
concentrations_symbols = ["X", "S", "PT", "PM", "V"]
concentrations_labels = [convert_symbol_to_label(symbol) for symbol in concentrations_symbols]

for symbol, label in zip((['t'] + concentrations_symbols), (['t'] + concentrations_labels)):
    max_val = max(data[label])
    min_val = min(data[label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    maxes_to_sbml[symbol] = max_val
    mins_to_sbml[symbol] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

#sd scaling
for symbol in concentrations_symbols:
    label_sd = convert_symbol_to_sd(symbol)
    label_sd_scaled = f"{label_sd}_scaled"
    label_concentration = convert_symbol_to_label(symbol)
    data[label_sd_scaled] = data[label_sd] / (maxes[label_concentration] - mins[label_concentration])

In [5]:
prev_val = 0
batches_lengths = []
for val in data["exp"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ["X", "S", "V", "Feed"]

def make_label_from_changing_var(v):
    if v == "Feed":
        return v
    else:
        return f"{v}_noise"

changing_vars_labels = [make_label_from_changing_var(changing_var) for changing_var in changing_vars]
first_of_each_batch = data[changing_vars_labels].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()

def make_constant_input_label(label):
    return f"{label}0"

mins_Xs_dict = {make_constant_input_label(label): minX for label, minX in zip(changing_vars, mins_Xs)}
maxes_Xs_dict = {make_constant_input_label(label): maxX for label, maxX in zip(changing_vars, maxes_Xs)}
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

Xs_ys_sds = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["t_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    sd = batch[[f"{convert_symbol_to_sd(symbol)}_scaled" for symbol in concentrations_symbols]]
    Xs_ys_sds.append((X, y, sd))

In [ ]:
INPUTS = 5
OUTPUTS = 5
class PRRegressor_ffnn(nn.Module):
    def __init__(self, inputs, layers, outputs, activation):
        super().__init__()
        all_layers = [inputs] + layers + [outputs]
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        if activation == "tanh":
            self.act = nn.Tanh()
        elif activation == "relu":
            self.act = nn.ReLU()
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
        x = self.linears[-1](x)
        return x

In [7]:
NUM_HIDDEN = 1
class PR_rnn(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, activation):
        super(PR_rnn, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity=activation)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

In [8]:
def common_sbml():
    document = libsbml.SBMLDocument(3, 1) 
    model = document.createModel() 
    model.setId("NN") 
    
    # Set default units in the SBML model 
    model.setLengthUnits("metre") 
    area_units = model.createUnitDefinition() 
    area_units.setId("m2") 
    unit = area_units.createUnit() 
    unit.setKind(libsbml.UNIT_KIND_METRE)  # Set kind to length 
    unit.setExponent(2)                      # Set exponent to 2 (for square meters) 
    unit.setScale(0)                         # Scale (0 means no scaling) 
    unit.setMultiplier(1)                    # Multiplier (1 means no change) 
    # Set default area units to the defined unit 
    model.setAreaUnits("m2") 
    model.setVolumeUnits("litre")  
    
    # Set model-wide units 
    model.setTimeUnits("second")          # Set model-wide time units to seconds 
    model.setExtentUnits("mole")        # Set model units of extent to moles 
    model.setSubstanceUnits("mole")     # Set model substance units to moles 
    
    # Define the unit for millimoles (mmol) 
    mmol = model.createUnitDefinition()  # Create unit definition for 
    mmol.setId("mmol")                   # Set unit definition id for mmol
    
    # Define 1 millimole = 10^-3 moles
    unit = mmol.createUnit()
    unit.setKind(libsbml.UNIT_KIND_MOLE)  # Set unit kind to mole 
    unit.setExponent(1)                   # Set unit exponent 
    unit.setScale(-3)                     # Set unit scale to -3 for millimoles 
    unit.setMultiplier(1)                 # Set unit multiplier to 1 
    
    model.setSubstanceUnits("mmol")     # Set model substance units to milimoles 
    model.setExtentUnits("mmol")        # Set model units of extent to milimoles 
    return document, model

In [9]:
def create_relu_function(model):
    relu_node = libsbml.parseL3Formula("piecewise(x, x>0, 0)")
    lambda_node = libsbml.ASTNode(libsbml.AST_LAMBDA)
    lambda_node.addChild(libsbml.parseL3Formula("x"))
    lambda_node.addChild(relu_node)
    relu = model.createFunctionDefinition()
    relu.setId("relu")
    relu.setMath(lambda_node)

In [ ]:
def big_assignment_rules_ffnn_tanh_sbml():
    def create_formula(inputs, weights, is_tanh=True):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        return formula
    def create_neuron(model, name, inputs, weights, is_tanh=True, scaling=False, compartment=""):
        formula = create_formula(inputs, weights, is_tanh=is_tanh)
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    if l == last:
                        sp_name = outputs_labels[m]
                        compartment = "c"
                        create_neuron(model,
                                    sp_name,
                                    inputs,
                                    weights,
                                    is_tanh=False,
                                    scaling=True,
                                    compartment=compartment)
                    else:
                        formula = create_formula(inputs,
                                                weights,
                                                is_tanh=True)
                        new_inputs.append(formula)
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def layers_as_compartments_ffnn_tanh_sbml():
    def create_neuron(model, name, compartment, inputs, weights, is_tanh=True, scaling=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        
        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                compartment_name = f"layer{k}"
                compartment = model.createCompartment() 
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                    else:
                        sp_name = f"n{k}_{mnd}"
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model, sp_name, compartment_name, inputs, weights, is_tanh=(l < last), scaling=(l == last))
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def nodes_as_parameters_ffnn_tanh_sbml():
    def create_neuron(model, name, inputs, weights, is_tanh=True, scaling=False, isOutput=False, compartment=""):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        if isOutput:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment(compartment)
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
        else:
            param = model.createParameter()
            param.setId(name)
            param.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                        isOutput = True
                        compartment = "c"
                    else:
                        sp_name = f"n{k}_{mnd}"
                        isOutput = False
                        compartment = ""
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model,
                                sp_name,
                                inputs,
                                weights,
                                is_tanh=(l < last),
                                scaling=(l == last),
                                isOutput=isOutput,
                                compartment=compartment)
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def nodes_as_species_ffnn_tanh_sbml():
    def create_neuron(model, name, compartment, inputs, weights, is_tanh=True, scaling=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                    else:
                        sp_name = f"n{k}_{mnd}"
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model, sp_name, "c", inputs, weights, is_tanh=(l < last), scaling=(l == last))
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def big_assignment_rules_ffnn_relu_sbml():
    def create_formula(inputs, weights, is_relu=True):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"piecewise(({formula}), ({formula}) > 0, 0)"
        return formula
    def create_neuron(model, name, inputs, weights, is_relu=True, scaling=False, compartment=""):
        formula = create_formula(inputs, weights, is_relu=is_relu)
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    if l == last:
                        sp_name = outputs_labels[m]
                        compartment = "c"
                        create_neuron(model,
                                    sp_name,
                                    inputs,
                                    weights,
                                    is_relu=False,
                                    scaling=True,
                                    compartment=compartment)
                    else:
                        formula = create_formula(inputs,
                                                weights,
                                                is_relu=True)
                        new_inputs.append(formula)
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def layers_as_compartments_ffnn_relu_sbml():
    def create_neuron(model, name, compartment, inputs, weights, is_relu=True, scaling=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        
        create_relu_function(model)

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                compartment_name = f"layer{k}"
                compartment = model.createCompartment() 
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                    else:
                        sp_name = f"n{k}_{mnd}"
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model, sp_name, compartment_name, inputs, weights, is_relu=(l < last), scaling=(l == last))
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def nodes_as_parameters_ffnn_relu_sbml():
    def create_neuron(model, name, inputs, weights, is_relu=True, scaling=False, isOutput=False, compartment=""):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        if isOutput:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment(compartment)
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
        else:
            param = model.createParameter()
            param.setId(name)
            param.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                        isOutput = True
                        compartment = "c"
                    else:
                        sp_name = f"n{k}_{mnd}"
                        isOutput = False
                        compartment = ""
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model,
                                sp_name,
                                inputs,
                                weights,
                                is_relu=(l < last),
                                scaling=(l == last),
                                isOutput=isOutput,
                                compartment=compartment)
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
def nodes_as_species_ffnn_relu_sbml():
    def create_neuron(model, name, compartment, inputs, weights, is_relu=True, scaling=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + [weights[-1]]
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        if scaling:
            formula = f"({formula})*({name}_max - {name}_min) + {name}_min"
        spec = model.createSpecies()
        spec.setId(name)
        spec.setCompartment(compartment)
        spec.setConstant(False) 
        spec.setBoundaryCondition(True) 
        spec.setHasOnlySubstanceUnits(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable(name)
        assignmentRule.setMath(libsbml.parseL3Formula(formula))
    def create_sbml(document_name: str,
                    neural_network_model: PRRegressor_ffnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        create_relu_function(model)
        
        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)
        
        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))
        
        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]
        k = 0
        last = len(list(neural_network_model.named_parameters())) - 1
        for l, param in enumerate(neural_network_model.named_parameters()):
            if param[0].endswith(".weight"):
                for i, line in enumerate(param[1]):
                    ind = i + 1
                    for j, p in enumerate(line):
                        jnd = j + 1
                        par = model.createParameter() 
                        par.setId(f"w{l}_{jnd}_{ind}") 
                        par.setValue(float(p)) 
                        par.setConstant(True)
            elif param[0].endswith(".bias"):
                new_inputs = []
                k += 1
                for m, p in enumerate(param[1]):
                    mnd = m + 1
                    if l == last:
                        sp_name = outputs_labels[m]
                    else:
                        sp_name = f"n{k}_{mnd}"
                    new_inputs.append(sp_name)
                    par = model.createParameter()
                    bias_name = f"w{l}_{mnd}"
                    par.setId(bias_name) 
                    par.setValue(float(p)) 
                    par.setConstant(True)
                    weights = [f"w{l-1}_{inp}_{mnd}" for inp in range(1, jnd+1)] + [bias_name]
                    create_neuron(model, sp_name, "c", inputs, weights, is_relu=(l < last), scaling=(l == last))
                inputs = new_inputs
        
        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [18]:
def big_event_assignments_ernn_tanh_sbml():
    def create_neuron_formula(inputs, weights, biases, is_tanh=True):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        else: formula = f"({formula})"
        return formula
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):

        document, model = common_sbml()


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])

        #copy event
        event1 = model.createEvent()
        event1.setId("event1")
        event1.setUseValuesFromTriggerTime(True)
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula("time>=t_event"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable("t_event")
        eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
        eventAssignment.setId("event1_time_assignment")



        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                
                # creating neurons
                new_inputs = []
                for i in range(len(param[1])):
                    joined_inputs = inputs + past_inputs
                    formula = create_neuron_formula(joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True)
                    new_inputs.append(formula)
                    # copy event assignment
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_inputs[i])
                    eventAssignment.setMath(libsbml.parseL3Formula(formula))
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                for i, output in enumerate(outputs_labels):
                    formula = create_neuron_formula(inputs, linear_weights[i], linear_biases[i], False)
                    par_name = "par_" + output
                    par = model.createParameter()
                    par.setId(par_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(par_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(formula))
                    spec = model.createSpecies()
                    spec.setId(output)
                    spec.setCompartment("c")
                    spec.setConstant(False) 
                    spec.setBoundaryCondition(True) 
                    spec.setHasOnlySubstanceUnits(False)
                    assRule = model.createAssignmentRule()
                    assRule.setVariable(output)
                    assRule.setMath(libsbml.parseL3Formula(f"{par_name} * ({output}_max - {output}_min) + {output}_min"))
                    assRule.setId(par_name + "_assignment_rule")

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [19]:
def layers_as_compartments_ernn_tanh_sbml():
    def create_neuron(model,
                    name,
                    event,
                    inputs,
                    weights,
                    biases,
                    compartment,
                    is_tanh=True,
                    is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        sp = model.createSpecies()
        pname = f"p_{name}"
        sp.setId(pname)
        sp.setCompartment(compartment)
        sp.setConstant(False) 
        sp.setBoundaryCondition(True) 
        sp.setHasOnlySubstanceUnits(False)
        pname = f"p_{name}"
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                compartment = model.createCompartment()
                compartment_name = f"layer_{layer_num + 1}"
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")

                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model,
                                        name,
                                        event,
                                        joined_inputs,
                                        weights_for_neurons[i],
                                        biases_for_neurons[i],
                                        compartment_name,
                                        True,
                                        False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                compartment_name = "output_layer"
                compartment = model.createCompartment()
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")

                for i, output in enumerate(outputs_labels):
                    create_neuron(model,
                                output,
                                event,
                                inputs,
                                linear_weights[i],
                                linear_biases[i],
                                compartment_name,
                                False,
                                True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [20]:
def nodes_as_parameters_ernn_tanh_sbml():
    def create_neuron(model, name, event, inputs, weights, biases, is_tanh=True, is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        par = model.createParameter()
        pname = f"p_{name}"
        par.setId(pname)
        par.setValue(0) 
        par.setConstant(False)
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model, name, event, joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True, False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i, output in enumerate(outputs_labels):
                    create_neuron(model, output, event, inputs, linear_weights[i], linear_biases[i], False, True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [21]:
def nodes_as_species_ernn_tanh_sbml():
    def create_neuron(model, name, event, inputs, weights, biases, is_tanh=True, is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_tanh:
            formula = f"tanh({formula})"
        sp = model.createSpecies()
        pname = f"p_{name}"
        sp.setId(pname)
        sp.setCompartment("c")
        sp.setConstant(False) 
        sp.setBoundaryCondition(True) 
        sp.setHasOnlySubstanceUnits(False)
        pname = f"p_{name}"
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model, name, event, joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True, False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i, output in enumerate(outputs_labels):
                    create_neuron(model, output, event, inputs, linear_weights[i], linear_biases[i], False, True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [22]:
def big_event_assignments_ernn_relu_sbml():
    def create_neuron_formula(inputs, weights, biases, is_relu=True):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"piecewise(({formula}), ({formula}) > 0, 0)"
        else: formula = f"({formula})"
        return formula
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):

        document, model = common_sbml()


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])

        #copy event
        event1 = model.createEvent()
        event1.setId("event1")
        event1.setUseValuesFromTriggerTime(True)
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula("time>=t_event"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable("t_event")
        eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
        eventAssignment.setId("event1_time_assignment")



        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                
                # creating neurons
                new_inputs = []
                for i in range(len(param[1])):
                    joined_inputs = inputs + past_inputs
                    formula = create_neuron_formula(joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True)
                    new_inputs.append(formula)
                    # copy event assignment
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_inputs[i])
                    eventAssignment.setMath(libsbml.parseL3Formula(formula))
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                for i, output in enumerate(outputs_labels):
                    formula = create_neuron_formula(inputs, linear_weights[i], linear_biases[i], False)
                    par_name = "par_" + output
                    par = model.createParameter()
                    par.setId(par_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(par_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(formula))
                    spec = model.createSpecies()
                    spec.setId(output)
                    spec.setCompartment("c")
                    spec.setConstant(False) 
                    spec.setBoundaryCondition(True) 
                    spec.setHasOnlySubstanceUnits(False)
                    assRule = model.createAssignmentRule()
                    assRule.setVariable(output)
                    assRule.setMath(libsbml.parseL3Formula(f"{par_name} * ({output}_max - {output}_min) + {output}_min"))
                    assRule.setId(par_name + "_assignment_rule")

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [23]:
def layers_as_compartments_ernn_relu_sbml():
    def create_neuron(model,
                    name,
                    event,
                    inputs,
                    weights,
                    biases,
                    compartment,
                    is_relu=True,
                    is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        sp = model.createSpecies()
        pname = f"p_{name}"
        sp.setId(pname)
        sp.setCompartment(compartment)
        sp.setConstant(False) 
        sp.setBoundaryCondition(True) 
        sp.setHasOnlySubstanceUnits(False)
        pname = f"p_{name}"
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        create_relu_function(model)


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                compartment = model.createCompartment()
                compartment_name = f"layer_{layer_num + 1}"
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")

                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model,
                                        name,
                                        event,
                                        joined_inputs,
                                        weights_for_neurons[i],
                                        biases_for_neurons[i],
                                        compartment_name,
                                        True,
                                        False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                compartment_name = "output_layer"
                compartment = model.createCompartment()
                compartment.setId(compartment_name)
                compartment.setConstant(True) 
                compartment.setSize(1.0) 
                compartment.setSpatialDimensions(3) 
                compartment.setUnits("litre")

                for i, output in enumerate(outputs_labels):
                    create_neuron(model,
                                output,
                                event,
                                inputs,
                                linear_weights[i],
                                linear_biases[i],
                                compartment_name,
                                False,
                                True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [24]:
def nodes_as_parameters_ernn_relu_sbml():
    def create_neuron(model, name, event, inputs, weights, biases, is_relu=True, is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        par = model.createParameter()
        pname = f"p_{name}"
        par.setId(pname)
        par.setValue(0) 
        par.setConstant(False)
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        create_relu_function(model)


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model, name, event, joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True, False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i, output in enumerate(outputs_labels):
                    create_neuron(model, output, event, inputs, linear_weights[i], linear_biases[i], False, True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [25]:
def nodes_as_species_ernn_relu_sbml():
    def create_neuron(model, name, event, inputs, weights, biases, is_relu=True, is_output=False):
        inputs_weights = [f"{inputs[i]}*{weights[i]}" for i in range(len(inputs))] + biases
        formula = "+".join(inputs_weights)
        if is_relu:
            formula = f"relu({formula})"
        sp = model.createSpecies()
        pname = f"p_{name}"
        sp.setId(pname)
        sp.setCompartment("c")
        sp.setConstant(False) 
        sp.setBoundaryCondition(True) 
        sp.setHasOnlySubstanceUnits(False)
        pname = f"p_{name}"
        eventAssignment = event.createEventAssignment()
        eventAssignment.setVariable(pname)
        eventAssignment.setMath(libsbml.parseL3Formula(formula))
        if is_output:
            spec = model.createSpecies()
            spec.setId(name)
            spec.setCompartment("c")
            spec.setConstant(False) 
            spec.setBoundaryCondition(True) 
            spec.setHasOnlySubstanceUnits(False)
            assignmentRule = model.createAssignmentRule()
            assignmentRule.setVariable(name)
            assignmentRule.setMath(libsbml.parseL3Formula(f"{pname} * ({name}_max - {name}_min) + {name}_min"))
        return pname
    def create_sbml(document_name: str,
                    neural_network_model: PR_rnn,
                    constant_inputs: Dict[str, float],
                    constant_inputs_mins: Dict[str, float],
                    constant_inputs_maxes: Dict[str, float],
                    t_max: float,
                    outputs_labels: List[str],
                    outputs_mins: Dict[str, float],
                    outputs_maxes: Dict[str, float]):
        document, model = common_sbml()
        create_relu_function(model)


        #add compartments
        #### Define the compartments (cytosol, mitochondria, ribosomal, nucleolus, extracellular) 
        #Compartment 1 
        compartment = model.createCompartment() 
        compartment.setId("c") # cytosol
        compartment.setConstant(True) 
        compartment.setSize(1.0) 
        compartment.setSpatialDimensions(3) 
        compartment.setUnits("litre")

        #scale factors
        for name, val in outputs_maxes.items():
            par = model.createParameter()
            par.setId(f"{name}_max")
            par.setValue(val) 
            par.setConstant(True)
        for name, val in outputs_mins.items():
            par = model.createParameter()
            par.setId(f"{name}_min")
            par.setValue(val) 
            par.setConstant(True)

        #scaled time
        par = model.createParameter() 
        par.setId("t_max")
        par.setValue(t_max) 
        par.setConstant(True)
        par = model.createParameter() 
        par.setId("t_scaled")
        par.setValue(0) 
        par.setConstant(False)
        assignmentRule = model.createAssignmentRule()
        assignmentRule.setVariable("t_scaled")
        assignmentRule.setMath(libsbml.parseL3Formula("time / t_max"))

        #input values
        for label, value in constant_inputs.items():
            par = model.createParameter() 
            par.setId(label)
            par.setValue(value) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_min")
            par.setValue(constant_inputs_mins[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_max")
            par.setValue(constant_inputs_maxes[label]) 
            par.setConstant(True)

            par = model.createParameter() 
            par.setId(f"{label}_scaled")
            par.setValue(value) 
            par.setConstant(True)

            initialAssignment = model.createInitialAssignment()
            initialAssignment.setSymbol(f"{label}_scaled")
            initialAssignment.setMath(libsbml.parseL3Formula(f"({label} - {label}_min) / ({label}_max - {label}_min)"))

        #event parameters
        par = model.createParameter() 
        par.setId("delay_event")
        par.setValue(1) 
        par.setConstant(True)

        par = model.createParameter() 
        par.setId("t_event")
        par.setValue(0) 
        par.setConstant(False)

        w_ih_l = "rnn.weight_ih_l"
        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                num_layers = layer_num + 1

        #copy event
        par = model.createParameter()
        par.setId(f"event_p_{num_layers}")
        par.setValue(0)
        par.setConstant(False)
        event1 = model.createEvent()
        event1.setUseValuesFromTriggerTime(True)
        event1.setId(f"event_{num_layers + 1}")
        trigger1 = event1.createTrigger()
        trigger1.setPersistent(True)
        trigger1.setInitialValue(False)
        trigger1.setMath(libsbml.parseL3Formula(f"event_p_{num_layers} == 1"))
        eventAssignment = event1.createEventAssignment()
        eventAssignment.setVariable(f"event_p_{num_layers}")
        eventAssignment.setMath(libsbml.parseL3Formula("0"))

        num_layers = 0
        # inner layers species
        w_ih_l = "rnn.weight_ih_l"
        b_ih_l = "rnn.bias_ih_l"
        w_hh_l = "rnn.weight_hh_l"
        b_hh_l = "rnn.bias_hh_l"

        inputs = ["t_scaled"] + [f"{label}_scaled" for label in constant_inputs.keys()]

        for param in neural_network_model.named_parameters():
            if param[0].startswith(w_ih_l):
                layer_num = int(param[0][len(w_ih_l):])
                weights_for_neurons = []
                for i, line in enumerate(param[1]):
                    weights_for_neurons.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_ih_{layer_num}_{j}_{i}"
                        weights_for_neurons[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(w_hh_l):
                layer_num = int(param[0][len(w_hh_l):])
                for i, line in enumerate(param[1]):
                    for j, weight in enumerate(line):
                        weight_name = f"w_hh_{layer_num}_{j}_{i}"
                        weights_for_neurons[i].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0].startswith(b_ih_l):
                layer_num = int(param[0][len(b_ih_l):])
                biases_for_neurons = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_ih_{layer_num}_{i}"
                    biases_for_neurons.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
            elif param[0].startswith(b_hh_l):
                layer_num = int(param[0][len(b_hh_l):])
                past_inputs = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_hh_{layer_num}_{i}"
                    biases_for_neurons[i].append(bias_name)
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)
                    
                    # creating "past" neurons to copy values
                    past_name = f"past_{layer_num}_{i}"
                    past_inputs.append(past_name)
                    par = model.createParameter()
                    par.setId(past_name)
                    par.setConstant(False) 
                    par.setValue(0)
                    eventAssignment = event1.createEventAssignment()
                    eventAssignment.setVariable(past_name)
                    eventAssignment.setMath(libsbml.parseL3Formula(f"p_n_{layer_num}_{i}"))
                
                # creating neurons
                new_inputs = []

                par = model.createParameter()
                par.setId(f"event_p_{layer_num}")
                par.setValue(0)
                par.setConstant(False)
                event = model.createEvent()
                event.setId(f"event_{layer_num}")
                event.setUseValuesFromTriggerTime(True)
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                if layer_num == 0:
                    trigger.setMath(libsbml.parseL3Formula("time>=t_event"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable("t_event")
                    eventAssignment.setMath(libsbml.parseL3Formula("t_event + delay_event"))
                else:
                    trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num - 1} == 1"))
                    eventAssignment = event.createEventAssignment()
                    eventAssignment.setVariable(f"event_p_{layer_num - 1}")
                    eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i in range(len(param[1])):
                    name = f"n_{layer_num}_{i}"
                    joined_inputs = inputs + past_inputs
                    pname = create_neuron(model, name, event, joined_inputs, weights_for_neurons[i], biases_for_neurons[i], True, False)
                    new_inputs.append(pname)
                inputs = new_inputs
            elif param[0] == "fc.weight":
                linear_weights = []
                for i, line in enumerate(param[1]):
                    linear_weights.append([])
                    for j, weight in enumerate(line):
                        weight_name = f"w_l_{j}_{i}"
                        linear_weights[-1].append(weight_name)
                        par = model.createParameter() 
                        par.setId(weight_name) 
                        par.setValue(float(weight)) 
                        par.setConstant(True)
            elif param[0] == "fc.bias":
                linear_biases = []
                for i, bias in enumerate(param[1]):
                    bias_name = f"b_l_{i}"
                    linear_biases.append([bias_name])
                    par = model.createParameter() 
                    par.setId(bias_name) 
                    par.setValue(float(bias)) 
                    par.setConstant(True)

                event = model.createEvent()
                event.setUseValuesFromTriggerTime(True)
                event.setId(f"event_{layer_num + 1}")
                trigger = event.createTrigger()
                trigger.setPersistent(True)
                trigger.setInitialValue(False)
                trigger.setMath(libsbml.parseL3Formula(f"event_p_{layer_num} == 1"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num}")
                eventAssignment.setMath(libsbml.parseL3Formula("0"))
                eventAssignment = event.createEventAssignment()
                eventAssignment.setVariable(f"event_p_{layer_num + 1}")
                eventAssignment.setMath(libsbml.parseL3Formula("1"))
                
                for i, output in enumerate(outputs_labels):
                    create_neuron(model, output, event, inputs, linear_weights[i], linear_biases[i], False, True)

        libsbml.writeSBMLToFile(document, document_name)
    return create_sbml

In [ ]:
# model_type can be "ffnn" or "ernn"
# activation can be "tanh" or "relu"
def load_model(model_name, path_to_folder, model_type, activation, layers):
    if model_type == "ffnn":
        nn_model = PRRegressor_ffnn(INPUTS, layers, OUTPUTS, activation)
    elif model_type == "ernn":
        nn_model = PR_rnn(INPUTS, layers, NUM_HIDDEN, OUTPUTS, activation)
    nn_model.load_state_dict(torch.load(os.path.join(path_to_folder, model_name), weights_only=True, map_location=torch.device('cpu')))
    nn_model.eval()
    return nn_model

In [27]:
funcs_dict = {
    "ffnn": {
        "tanh": {
            "big_assignment_rules": big_assignment_rules_ffnn_tanh_sbml(),
            "layers_as_compartments": layers_as_compartments_ffnn_tanh_sbml(),
            "nodes_as_parameters": nodes_as_parameters_ffnn_tanh_sbml(),
            "nodes_as_species": nodes_as_species_ffnn_tanh_sbml()
        },
        "relu": {
            "big_assignment_rules": big_assignment_rules_ffnn_relu_sbml(),
            "layers_as_compartments": layers_as_compartments_ffnn_relu_sbml(),
            "nodes_as_parameters": nodes_as_parameters_ffnn_relu_sbml(),
            "nodes_as_species": nodes_as_species_ffnn_relu_sbml()
        }
    },
    "ernn" : {
        "tanh": {
            "big_event_assignments": big_event_assignments_ernn_tanh_sbml(),
            "layers_as_compartments": layers_as_compartments_ernn_tanh_sbml(),
            "nodes_as_parameters": nodes_as_parameters_ernn_tanh_sbml(),
            "nodes_as_species": nodes_as_species_ernn_tanh_sbml()
        },
        "relu": {
            "big_event_assignments": big_event_assignments_ernn_relu_sbml(),
            "layers_as_compartments": layers_as_compartments_ernn_relu_sbml(),
            "nodes_as_parameters": nodes_as_parameters_ernn_relu_sbml(),
            "nodes_as_species": nodes_as_species_ernn_relu_sbml()
        }
    }
}

In [28]:
model_names_ffnn_tanh = ["PR_ffnn_tanh_wmse_13_13_best",
                         "PR_ffnn_tanh_wmse_15_15_best",
                         "PR_ffnn_tanh_wmse_13_13_13_best"]
path_to_folder_ffnn_tanh = "../pr_best_models/"
model_names_ffnn_relu = ["PR_ffnn_relu_wmse_13_13_best",
                         "PR_ffnn_relu_wmse_15_15_best",
                         "PR_ffnn_relu_wmse_13_13_13_best"]
path_to_folder_ffnn_relu = "../pr_best_models/"
model_names_rnn_tanh = ["PR_rnn_tanh_wmse_10_best",
                        "PR_rnn_tanh_wmse_15_best",
                        "PR_rnn_tanh_wmse_19_best"]
path_to_folder_rnn_tanh = "../pr_best_models/"
model_names_rnn_relu = ["PR_rnn_relu_wmse_10_best",
                        "PR_rnn_relu_wmse_15_best",
                        "PR_rnn_relu_wmse_19_best"]
path_to_folder_rnn_relu = "../pr_best_models/"
layers_ffnn = [[13, 13], [15, 15], [13, 13, 13]]
layers_ernn = [10, 15, 19]
layers_dict = {"ffnn": layers_ffnn, "ernn": layers_ernn}
models_dict = {"ffnn": {"tanh": (model_names_ffnn_tanh, path_to_folder_ffnn_tanh),
                        "relu": (model_names_ffnn_relu, path_to_folder_ffnn_relu)},
               "ernn": {"tanh": (model_names_rnn_tanh, path_to_folder_rnn_tanh),
                        "relu": (model_names_rnn_relu, path_to_folder_rnn_relu)}}

In [29]:
BATCH_NUMBER = 0
this_X = batches[BATCH_NUMBER]
constant_inputs = {make_constant_input_label(label_changing): this_X[make_label_from_changing_var(label_changing)].iloc[0] for label_changing in changing_vars}
t_max = float(data["t"].max())
outputs_labels = concentrations_symbols
outputs_mins = {label_changing: mins_to_sbml[label_changing] for label_changing in outputs_labels}
outputs_maxes = {label_changing: maxes_to_sbml[label_changing] for label_changing in outputs_labels}

In [30]:
for model_type, layers_s in layers_dict.items():
    models_tanh_relu = models_dict[model_type]
    for activation, (model_names, path_to_folder) in models_tanh_relu.items():
        for model_name, layers in zip(model_names, layers_s):
            if type(layers) is list:
                layers_string = "_".join([str(l) for l in layers])
            else:
                layers_string = str(layers)
            nn_model = load_model(model_name, path_to_folder, model_type, activation, layers)
            for encoding_type, encoding_fun in funcs_dict[model_type][activation].items():
                document_name = f"PR_{model_type}_{activation}_{layers_string}_{encoding_type}_sbml.xml"

                encoding_fun(document_name,
                            nn_model,
                            constant_inputs,
                            mins_Xs_dict,
                            maxes_Xs_dict,
                            t_max,
                            outputs_labels,
                            outputs_mins,
                            outputs_maxes)